# Step 5: Load Triples into Neo4j

This notebook loads all knowledge graph triples into Neo4j:
1. Load USDA triples (all parts)
2. Load PubMed triples with evidence metadata
3. Create necessary nodes, relationships, and indexes

In [2]:
# Import required libraries
import pandas as pd
import os
import time
from neo4j import GraphDatabase

In [3]:
# Neo4j Configuration
NEO4J_URI = "neo4j://127.0.0.1:7687"  
NEO4J_USER = "neo4j"                 
NEO4J_PASSWORD = "password"           

# Data paths
KG_DATA_PATH = '../data/pre-processed/kg/'

# Batch size for loading
BATCH_SIZE = 10000

print(f"Neo4j URI: {NEO4J_URI}")
print(f"KG Data Path: {KG_DATA_PATH}")

Neo4j URI: neo4j://127.0.0.1:7687
KG Data Path: ../data/pre-processed/kg/


In [4]:
# Connect to Neo4j
print("Connecting to Neo4j...")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Test connection
with driver.session() as session:
    result = session.run("RETURN 1 AS connected")
    if result.single()["connected"] == 1:
        print("✅ Connected to Neo4j successfully!")
    else:
        print("❌ Connection test failed")

Connecting to Neo4j...
✅ Connected to Neo4j successfully!


## 1. Setup: Clear Data and Create Indexes

In [5]:
# Clear existing data (optional - set to False to keep existing data)
CLEAR_EXISTING = True

with driver.session() as session:
    if CLEAR_EXISTING:
        print("Clearing existing data...")
        session.run("MATCH (n) DETACH DELETE n")
        print("✅ Cleared all existing data")
    
    # Create indexes for faster lookups
    print("\nCreating indexes...")
    session.run("CREATE INDEX IF NOT EXISTS FOR (f:Food) ON (f.id)")
    session.run("CREATE INDEX IF NOT EXISTS FOR (f:Food) ON (f.name)")
    session.run("CREATE INDEX IF NOT EXISTS FOR (b:Brand) ON (b.name)")
    session.run("CREATE INDEX IF NOT EXISTS FOR (i:Ingredient) ON (i.name)")
    session.run("CREATE INDEX IF NOT EXISTS FOR (d:Disease) ON (d.name)")
    print("✅ Indexes created")

Clearing existing data...
✅ Cleared all existing data

Creating indexes...
✅ Indexes created


## 2. Load USDA Triples

In [6]:
# Find all USDA triple files
usda_files = sorted([f for f in os.listdir(KG_DATA_PATH) if f.startswith('triples_part_') and f.endswith('.csv')])
print(f"Found {len(usda_files)} USDA triple files:")
for f in usda_files:
    size = os.path.getsize(os.path.join(KG_DATA_PATH, f)) / (1024*1024)
    print(f"  - {f}: {size:.1f} MB")

Found 5 USDA triple files:
  - triples_part_1.csv: 233.1 MB
  - triples_part_2.csv: 239.7 MB
  - triples_part_3.csv: 239.9 MB
  - triples_part_4.csv: 244.3 MB
  - triples_part_5.csv: 29.5 MB


In [7]:
def load_usda_batch(tx, has_name_data, has_brand_data, contains_ing_data):
    """Load USDA triples in batches using UNWIND for efficiency."""
    
    # Load has_name triples (create Food nodes)
    if has_name_data:
        tx.run("""
            UNWIND $data AS row
            MERGE (f:Food {id: row.sub})
            ON CREATE SET f.name = row.obj
        """, data=has_name_data)
    
    # Load has_brand triples
    if has_brand_data:
        tx.run("""
            UNWIND $data AS row
            MERGE (f:Food {id: row.sub})
            MERGE (b:Brand {name: row.obj})
            MERGE (f)-[:HAS_BRAND]->(b)
        """, data=has_brand_data)
    
    # Load contains_ingredient triples
    if contains_ing_data:
        tx.run("""
            UNWIND $data AS row
            MERGE (f:Food {id: row.sub})
            MERGE (i:Ingredient {name: row.obj})
            MERGE (f)-[:CONTAINS_INGREDIENT]->(i)
        """, data=contains_ing_data)

In [9]:
# Load each USDA file
total_start = time.time()
total_loaded = 0
skipped_records = {'has_name': 0, 'has_brand': 0, 'contains_ingredient': 0}

for file_idx, filename in enumerate(usda_files):
    filepath = os.path.join(KG_DATA_PATH, filename)
    print(f"\n{'='*50}")
    print(f"Loading {filename} ({file_idx+1}/{len(usda_files)})...")
    
    # Read CSV
    df = pd.read_csv(filepath)
    print(f"Loaded {len(df)} triples")
    
    # Separate by predicate type
    df_has_name = df[df['Predicate'] == 'has_name']
    df_has_brand = df[df['Predicate'] == 'has_brand']
    df_contains_ing = df[df['Predicate'] == 'contains_ingredient']
    
    print(f"  has_name: {len(df_has_name)}, has_brand: {len(df_has_brand)}, contains_ingredient: {len(df_contains_ing)}")
    
    file_start = time.time()
    
    with driver.session() as session:
        # Process each predicate type
        for pred_name, pred_df in [('has_name', df_has_name), ('has_brand', df_has_brand), ('contains_ingredient', df_contains_ing)]:
            if len(pred_df) == 0:
                continue
            
            # Filter out rows with null/NaN in Subject or Object columns
            pred_df_clean = pred_df.dropna(subset=['Subject', 'Object'])
            
            # Also filter out empty strings
            pred_df_clean = pred_df_clean[
                (pred_df_clean['Subject'].astype(str).str.strip() != '') & 
                (pred_df_clean['Object'].astype(str).str.strip() != '')
            ]
            
            skipped = len(pred_df) - len(pred_df_clean)
            if skipped > 0:
                skipped_records[pred_name] += skipped
                print(f"  ⚠️  Skipped {skipped} invalid {pred_name} records (null/NaN/empty values)")
                
            for i in range(0, len(pred_df_clean), BATCH_SIZE):
                batch = pred_df_clean.iloc[i:i+BATCH_SIZE]
                batch_data = [{'sub': r['Subject'], 'obj': r['Object']} for _, r in batch.iterrows()]
                
                # Prepare data for the right type
                if pred_name == 'has_name':
                    session.execute_write(load_usda_batch, batch_data, [], [])
                elif pred_name == 'has_brand':
                    session.execute_write(load_usda_batch, [], batch_data, [])
                else:
                    session.execute_write(load_usda_batch, [], [], batch_data)
                
                total_loaded += len(batch)
                
                if (i // BATCH_SIZE + 1) % 10 == 0:
                    elapsed = time.time() - file_start
                    print(f"    {pred_name}: {i+len(batch)}/{len(pred_df_clean)} ({elapsed:.1f}s)")
    
    file_time = time.time() - file_start
    print(f"  ✅ Completed in {file_time:.1f}s")

total_time = time.time() - total_start
print(f"\n{'='*50}")
print(f"USDA Loading Complete!")
print(f"Total triples loaded: {total_loaded:,}")
print(f"Skipped records: {sum(skipped_records.values()):,}")
print(f"  - has_name: {skipped_records['has_name']:,}")
print(f"  - has_brand: {skipped_records['has_brand']:,}")
print(f"  - contains_ingredient: {skipped_records['contains_ingredient']:,}")
print(f"Total time: {total_time/60:.1f} minutes")


Loading triples_part_1.csv (1/5)...
Loaded 5000000 triples
  has_name: 480385, has_brand: 480385, contains_ingredient: 4039230
  ⚠️  Skipped 1 invalid has_name records (null/NaN/empty values)
    has_name: 100000/480384 (6.0s)
    has_name: 200000/480384 (11.5s)
    has_name: 300000/480384 (17.3s)
    has_name: 400000/480384 (23.9s)
    has_brand: 100000/480385 (35.3s)
    has_brand: 200000/480385 (42.0s)
    has_brand: 300000/480385 (48.6s)
    has_brand: 400000/480385 (55.0s)
  ⚠️  Skipped 7 invalid contains_ingredient records (null/NaN/empty values)
    contains_ingredient: 100000/4039223 (69.1s)
    contains_ingredient: 200000/4039223 (75.9s)
    contains_ingredient: 300000/4039223 (82.3s)
    contains_ingredient: 400000/4039223 (88.6s)
    contains_ingredient: 500000/4039223 (94.9s)
    contains_ingredient: 600000/4039223 (101.6s)
    contains_ingredient: 700000/4039223 (108.8s)
    contains_ingredient: 800000/4039223 (115.8s)
    contains_ingredient: 900000/4039223 (121.9s)
    

## 3. Load PubMed Triples

In [10]:
# Load PubMed triples
pubmed_file = os.path.join(KG_DATA_PATH, 'pubmed_kg_triples.csv')

# Check if the new file exists, otherwise use the old one
if not os.path.exists(pubmed_file):
    pubmed_file = os.path.join(KG_DATA_PATH, 'pubmed_triples.xls')

print(f"Loading PubMed triples from {pubmed_file}...")
pubmed_df = pd.read_csv(pubmed_file, sep=',', on_bad_lines='warn', low_memory=False)
print(f"Loaded {len(pubmed_df)} PubMed triples")
print(f"Columns: {pubmed_df.columns.tolist()}")
pubmed_df.head()

Loading PubMed triples from ../data/pre-processed/kg/pubmed_kg_triples.csv...
Loaded 30567 PubMed triples
Columns: ['subject', 'predicate', 'object', 'pmid', 'title', 'year', 'journal']


,subject,predicate,object,pmid,title,year,journal
0,protein,RELATES_TO,"diabetes, insulin resistance, cardiovascular d...",41390803,Association between triglyceride-glucose index...,2025,Cardiovascular diabetology
1,protein,RELATES_TO,alzheimer,41390778,Multi-task learning identifies shared genetic ...,2025,Scientific reports
2,protein,RELATES_TO,"diabetes, obesity, diabetes mellitus",41390701,Obesity concurrent with gestational diabetes m...,2025,Scientific reports
3,protein,RELATES_TO,"diabetes, type 2 diabetes",41390575,The RNA-binding protein CPEB1 marks healthy ad...,2025,Scientific reports
4,protein,RELATES_TO,"inflammation, cardiovascular disease, stroke, ...",41390442,Genetic insights into 5-LOX-activating protein...,2025,Human genomics


In [11]:
def load_pubmed_batch(tx, data):
    """Load PubMed ingredient-disease relationships with evidence."""
    tx.run("""
        UNWIND $data AS row
        MERGE (i:Ingredient {name: row.ingredient})
        MERGE (d:Disease {name: row.disease})
        MERGE (i)-[r:RELATES_TO]->(d)
        ON CREATE SET r.pmid = row.pmid, r.title = row.title, r.year = row.year, r.journal = row.journal
    """, data=data)

In [12]:
# Determine column names
ing_col = 'subject' if 'subject' in pubmed_df.columns else 'ingredient'
dis_col = 'object' if 'object' in pubmed_df.columns else 'disease'

print(f"Using columns: ingredient='{ing_col}', disease='{dis_col}'")
print(f"Loading PubMed triples into Neo4j...")

pubmed_start = time.time()
pubmed_loaded = 0

with driver.session() as session:
    for i in range(0, len(pubmed_df), BATCH_SIZE):
        batch = pubmed_df.iloc[i:i+BATCH_SIZE]
        
        batch_data = []
        for _, r in batch.iterrows():
            ingredient = str(r.get(ing_col, '')).strip()
            disease = str(r.get(dis_col, '')).strip()
            
            if ingredient and disease and ingredient.lower() not in ['nan', ''] and disease.lower() not in ['nan', '']:
                batch_data.append({
                    'ingredient': ingredient,
                    'disease': disease,
                    'pmid': str(r.get('pmid', '')),
                    'title': str(r.get('title', ''))[:500],  # Truncate long titles
                    'year': str(r.get('year', '')),
                    'journal': str(r.get('journal', ''))
                })
        
        if batch_data:
            session.execute_write(load_pubmed_batch, batch_data)
            pubmed_loaded += len(batch_data)
        
        if (i // BATCH_SIZE + 1) % 5 == 0:
            elapsed = time.time() - pubmed_start
            print(f"  Loaded {pubmed_loaded:,} triples ({elapsed:.1f}s)")

pubmed_time = time.time() - pubmed_start
print(f"\n✅ PubMed loading complete!")
print(f"Total PubMed triples loaded: {pubmed_loaded:,}")
print(f"Time: {pubmed_time:.1f}s")

Using columns: ingredient='subject', disease='object'
Loading PubMed triples into Neo4j...

✅ PubMed loading complete!
Total PubMed triples loaded: 30,567
Time: 4.1s


## 4. Verify Loading

In [13]:
# Verify node and relationship counts
print("\nVerifying loaded data...")

with driver.session() as session:
    # Node counts
    print("\nNode Counts:")
    nodes_result = session.run("MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC")
    for record in nodes_result:
        print(f"  {record['label']}: {record['count']:,}")
    
    # Relationship counts
    print("\nRelationship Counts:")
    rels_result = session.run("MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC")
    for record in rels_result:
        print(f"  {record['type']}: {record['count']:,}")


Verifying loaded data...

Node Counts:
  ['Food']: 1,977,398
  ['Ingredient']: 225,440
  ['Brand']: 37,033
  ['Disease']: 2,324

Relationship Counts:
  CONTAINS_INGREDIENT: 16,353,259
  HAS_BRAND: 1,977,398
  RELATES_TO: 14,242


In [14]:
# Summary
print("\n" + "="*50)
print("Step 5: Load Triples into Neo4j - COMPLETE")
print("="*50)
print(f"\nUSDA triples loaded: {total_loaded:,}")
print(f"PubMed triples loaded: {pubmed_loaded:,}")
print(f"\nTotal time: {(time.time() - total_start)/60:.1f} minutes")

driver.close()
print("\n✅ Neo4j connection closed")


Step 5: Load Triples into Neo4j - COMPLETE

USDA triples loaded: 20,598,499
PubMed triples loaded: 30,567

Total time: 26.4 minutes

✅ Neo4j connection closed
